In [3]:
import os
import subprocess
from pathlib import Path

import cv2
import imageio_ffmpeg
from dotenv import load_dotenv
from ultralytics import solutions

load_dotenv()

# Folders
MODELS_DIR = Path("models")
INPUTS_DIR = Path("inputs")
OUTPUTS_DIR = Path("outputs")
for folder in (MODELS_DIR, INPUTS_DIR, OUTPUTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Paths & settings
model_name = os.getenv("MODEL_WEIGHTS", "yolo26s.pt")
model_path = MODELS_DIR / Path(model_name).name  # saves/loads inside models/

input_video = Path(os.getenv("SOURCE", str(INPUTS_DIR / "video.mp4")))
if not input_video.exists():
    raise FileNotFoundError(f"Input video not found: {input_video}")

temp_output = OUTPUTS_DIR / f"{input_video.stem}_temp.mp4"
final_output = OUTPUTS_DIR / f"{input_video.stem}_visioneye.mp4"

cap = cv2.VideoCapture(str(input_video))
assert cap.isOpened(), f"Error reading video file: {input_video}"

w, h, fps = (
    int(cap.get(x))
    for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS)
)
fps = fps if fps > 0 else 30

# Temporary OpenCV writer (will be transcoded to H.264 for LinkedIn/WhatsApp)
video_writer = cv2.VideoWriter(
    str(temp_output),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

# Initialize VisionEye (top-left corner + 3 classes: 0=person, 1=bicycle, 2=car)
visioneye = solutions.VisionEye(
    show=True,
    model=str(model_path),
    classes=[0, 1, 2],
    vision_point=(30, 50),  # Top-left corner
    conf=float(os.getenv("CONFIDENCE_THRESHOLD", 0.25)),
    iou=float(os.getenv("IOU_THRESHOLD", 0.45)),
)

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video processing completed.")
        break

    results = visioneye(im0)
    video_writer.write(results.plot_im)

cap.release()
video_writer.release()
cv2.destroyAllWindows()

# Convert to H.264 (yuv420p + faststart) for LinkedIn & WhatsApp compatibility
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
subprocess.run(
    [
        ffmpeg_exe,
        "-y",
        "-i",
        str(temp_output),
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        str(final_output),
    ],
    check=True,
)
temp_output.unlink(missing_ok=True)
print(f"Saved WhatsApp/LinkedIn-ready video to: {final_output}")

Ultralytics Solutions:  {'source': None, 'model': 'models\\yolo26s.pt', 'classes': [0, 1, 2], 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90.0, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (30, 50), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.45, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'tracktrack.yaml', 'verbose': True, 'data': 'images', 'recognizer': 'paddleocr'}
0: 640x640 87.7ms, 5 person
Speed: 2306.7ms track, 87.7ms solution per image at shape (1, 3, 640, 640)

1: 640x640 8.5ms, 5 person
Speed: 122.5ms track, 8.5ms solution per image at shape (1, 3, 640, 640)

2: 640x640 4.2ms, 5 person
Speed: 209.8ms track, 4.2ms solution per 